# Ridge Regression — Statistics

**Goal.** Treat the ridge estimator $\hat{\theta}_{ridge}$ as a random vector. Derive its bias, variance, and MSE; prove the classical **Hoerl–Kennard theorem** (1970) that there always exists some $\lambda$ > 0 at which ridge has *strictly lower* MSE than OLS; quantify the model complexity via the *effective degrees of freedom* defined in `02_mathematics.ipynb` §4; and turn all of this into a concrete $\lambda$-selection rule (k-fold CV, the LOO-CV closed form, generalised cross-validation).

**Role of this notebook.** Estimator theory: math first, with minimal Monte Carlo simulations used only to *verify* a theorem or visualise the trade-off. The Gauss–Markov machinery of `01_linear_regression/04_statistics.ipynb` is the starting point — ridge breaks unbiasedness on purpose, and the question becomes *when* that trade is worth it.

**Prerequisites.** `01_linear_regression/04_statistics.ipynb` (the bias / variance / sampling-identity framework); `02_mathematics.ipynb` of this folder (the closed form, the SVD shrinkage, the effective dof).

**Stage map.** `01_intuition` → `02_mathematics` → `03_optimization` → **`04_statistics`** → `05_hands_on_programming`.

**Seven questions.**

1. What does the sampling identity say about ridge?
2. Why is ridge **biased**, and what is its variance?
3. Can a biased estimator beat an unbiased one? (Yes — Hoerl–Kennard 1970.)
4. How does the MSE move with $\lambda$, and how do we *see* the trade-off in numbers?
5. How do we pick $\lambda$ without peeking at test data? (k-fold and LOO-CV.)
6. What is the closed-form LOO-CV for ridge?
7. What is generalised cross-validation (GCV)?

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import random

import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## 1. The ridge sampling identity

Setup (same as `01_linear_regression/04_statistics.ipynb` §1):

- Fixed design $X \in \mathbb{R}^{n \times p}$, full rank assumed only where stated.
- True parameter $\theta \in \mathbb{R}^p$.
- y = $X \theta$ + $\varepsilon$ with $\varepsilon$ satisfying A1 + A2 + A3 + A4 (𝔼[$\varepsilon$] = 0, Var($\varepsilon$) = $\sigma^2$ $I_n$).

Define the **ridge shrinkage matrix**

> $A_{\lambda}$  :=  ($X^T X$ + n $\lambda$ $I_p$)^{-1} $X^T X$  $\in$ $\mathbb{R}^{p \times p}$.   (1.1)

Substituting y = $X \theta$ + $\varepsilon$ into the ridge closed form (2.2) of `02_mathematics.ipynb` and simplifying:

```
$\hat{\theta}_{ridge}$  =  ($X^T X$ + n $\lambda$ $I_p$)^{-1} $X^T y$
          =  ($X^T X$ + n $\lambda$ $I_p$)^{-1} Xᵀ ($X \theta$ + $\varepsilon$)
          =  ($X^T X$ + n $\lambda$ $I_p$)^{-1} $X^T X$ $\cdot$ $\theta$   +   ($X^T X$ + n $\lambda$ $I_p$)^{-1} Xᵀ $\cdot$ $\varepsilon$
          =  $A_{\lambda}$ $\cdot$ $\theta$                    +   ($X^T X$ + n $\lambda$ $I_p$)^{-1} Xᵀ $\cdot$ $\varepsilon$.       (1.2)
```

(1.2) is the **ridge sampling identity**. Compare with the OLS sampling identity (3.1 of `01_linear_regression/04_statistics.ipynb`): the OLS version is $\theta$ + ($X^T X$)^{-1} Xᵀ $\varepsilon$, with the first term equal to the truth. The ridge version has the truth multiplied by $A_{\lambda}$ — and **$A_{\lambda}$ $\neq$ $I_p$** unless $\lambda$ = 0. That single fact is the source of all of ridge's bias.

## 2. Bias and variance of $\hat{\theta}_{ridge}$

### 2.1 Theorem (bias)

> **Theorem 2.1.** Under A1 + A2,
>
> 𝔼[ $\hat{\theta}_{ridge}$ ]  =  $A_{\lambda}$ $\theta$,    and    Bias($\hat{\theta}_{ridge}$)  =  ($A_{\lambda}$ - $I_p$) $\theta$  =  - n $\lambda$ $\cdot$ ($X^T X$ + n $\lambda$ $I_p$)^{-1} $\theta$.

**Proof.** Take expectation of (1.2) using 𝔼[$\varepsilon$] = 0 (A2):

```
𝔼[$\hat{\theta}_{ridge}$]  =  $A_{\lambda}$ $\theta$  +  0  =  $A_{\lambda}$ $\theta$.
```

For the bias formula use $A_{\lambda}$ - $I_p$ = ($X^T X$ + n $\lambda$ $I_p$)^{-1} $X^T X$ - $I_p$ = - n $\lambda$ $\cdot$ ($X^T X$ + n $\lambda$ $I_p$)^{-1}. ∎

**Reading.** The bias is non-zero whenever $\lambda$ > 0 and $\theta$ $\neq$ 0. Direction: every coordinate of $\theta$ is *shrunk* toward zero (the bias points back toward the origin). Magnitude: scales as n $\lambda$ $\cdot$ $\|\theta\|$ for small $\lambda$, then saturates at - $\theta$ as $\lambda$ → $\infty$ (where 𝔼[$\hat{\theta}_{ridge}$] → 0).

### 2.2 Theorem (variance)

> **Theorem 2.2.** Under A1 + A2 + A3 + A4,
>
> Var($\hat{\theta}_{ridge}$)  =  $\sigma^2$ $\cdot$ ($X^T X$ + n $\lambda$ $I_p$)^{-1} $X^T X$ ($X^T X$ + n $\lambda$ $I_p$)^{-1}.   (2.1)

**Proof.** From (1.2), $\hat{\theta}_{ridge}$ - 𝔼[$\hat{\theta}_{ridge}$] = ($X^T X$ + n $\lambda$ $I_p$)^{-1} Xᵀ $\varepsilon$. The general rule Var(M$\varepsilon$) = M Var($\varepsilon$) Mᵀ with Var($\varepsilon$) = $\sigma^2$ $I_n$ (A3 + A4) gives

```
Var($\hat{\theta}_{ridge}$)  =  $\sigma^2$ $\cdot$ ($X^T X$ + n $\lambda$ $I_p$)^{-1} Xᵀ $\cdot$ $I_n$ $\cdot$ X ($X^T X$ + n $\lambda$ $I_p$)^{-1}
                =  $\sigma^2$ $\cdot$ ($X^T X$ + n $\lambda$ $I_p$)^{-1} $\cdot$ $X^T X$ $\cdot$ ($X^T X$ + n $\lambda$ $I_p$)^{-1}.    ∎
```

### 2.3 SVD form (matches `02_mathematics.ipynb` §3)

Using X = U $\Sigma$ $V^T$:

> Var($\hat{\theta}_{ridge}$)  =  $\sigma^2$ $\cdot$ V $\cdot$ diag( $\sigma_j$^2 / ($\sigma_j$^2 + n $\lambda$)^2 ) $\cdot$ $V^T$.   (2.2)

The j-th eigenvalue of Var($\hat{\theta}_{ridge}$) is $\sigma^2$ $\cdot$ $\sigma_j$^2 / ($\sigma_j$^2 + n $\lambda$)^2. Compare with OLS (`01_linear_regression/04_statistics.ipynb` (4.1)): $\sigma^2$ / $\sigma_j$^2. The ratio (ridge / OLS) equals

> $\sigma_j$^4 / ($\sigma_j$^2 + n $\lambda$)^2  =  $\rho_{j}$($\lambda$)^2  $\in$  (0, 1],

where $\rho_{j}$($\lambda$) is the shrinkage factor (3.3) of `02_mathematics.ipynb`. **Ridge variance is everywhere $\le$ OLS variance.** The decrease is largest for small $\sigma_j$ — exactly the noisy directions where OLS variance was worst.

## 3. Total MSE — and why a biased estimator can win

The MSE of a *vector* estimator decomposes coordinate-wise as bias^2 + variance (vector version of Theorem 2.1 of `02_polynomial_regression/04_statistics.ipynb`).

### 3.1 Theorem (ridge MSE in SVD coordinates)

Define the **expected total squared error**

> MSE($\lambda$)  :=  𝔼[ $\|\hat{\theta}_{\mathrm{ridge}} - \theta\|^2$ ]
>          =  $\|\operatorname{Bias}(\hat{\theta}_{\mathrm{ridge}})\|^2$  +  trace( Var($\hat{\theta}_{ridge}$) ).

Substituting Theorems 2.1 and 2.2 (in the SVD basis) and writing $\alpha$ := $V^T$ $\theta$ for the true parameter expressed in V-coordinates:

> MSE($\lambda$)  =  $\sum_{j=1}^{p}$  [ ( n $\lambda$ / ($\sigma_j$^2 + n $\lambda$) )^2 $\cdot$ $\alpha_{j}$^2   +   $\sigma^2$ $\cdot$ $\sigma_j$^2 / ($\sigma_j$^2 + n $\lambda$)^2 ].   (3.1)

### 3.2 Theorem (Hoerl–Kennard 1970)

> **Theorem 3.2 (Hoerl & Kennard 1970).** For every X with $\text{rank}(X)$ = p, every $\theta$ $\neq$ 0, and every $\sigma^2$ > 0, there exists $\lambda$\* > 0 such that
>
> MSE($\lambda$\*)  <  MSE(0)  =  $\sigma^2$ $\cdot$ $\sum_{j}$ 1 / $\sigma_j$^2.

**Proof sketch.** Compute the derivative of (3.1) with respect to $\lambda$ at $\lambda$ = 0:

```
d MSE / d $\lambda$ |_{$\lambda$=0}  =  - 2 $\sigma^2$ n $\cdot$ $\sum_{j}$ 1 / $\sigma_j$^2  <  0.
```

The bias^2 term contributes 0 at $\lambda$ = 0 (the bias itself is 0) so all the action is in the variance reduction. Since MSE($\lambda$) is differentiable, a strictly negative derivative at $\lambda$ = 0 forces MSE to drop on a neighborhood of 0 — at any small $\lambda$ > 0 the MSE is strictly less than MSE(0). ∎

*(For the explicit optimal $\lambda$\*, see Hoerl & Kennard, "Ridge regression: biased estimation for nonorthogonal problems", Technometrics 12 (1970). The argmin has a complicated closed form depending on $\theta$ and $\sigma^2$, but the theorem only needs *existence*.)*

### 3.3 Reading

This is the central theoretical justification for ridge: **a small amount of regularisation always pays off**, in the squared-error metric. Gauss–Markov (Theorem 5.2 of `01_linear_regression/04_statistics.ipynb`) said OLS is the best *linear unbiased* estimator. Ridge is linear (in y) but *biased* — so it doesn't violate Gauss–Markov. It also outperforms OLS in MSE — so it shows that the "unbiased" requirement of Gauss–Markov was costing us something.

**Caveat.** The theorem only guarantees existence of a good $\lambda$ — it does not tell us *which* $\lambda$. The argmin depends on the unknown $\theta$ and $\sigma^2$. We need a data-driven procedure for that — cross-validation, next.

### 3.4 Monte Carlo verification

Pick a true $\theta$ and $\sigma^2$, draw many fresh y's, fit ridge at a sweep of $\lambda$. Plot empirical squared bias, variance, and total MSE against $\lambda$ — verify the U-shape and the existence of an interior minimum below MSE(0).

In [ ]:
n, p = 100, 12
rng_mc = np.random.default_rng(SEED)
X = rng_mc.normal(size=(n, p))
theta = rng_mc.normal(size=p)
sigma = 0.7

lams = np.logspace(-4, 2, 40)
M = 400

estimates = np.empty((len(lams), M, p))
for m in range(M):
    y = X @ theta + rng_mc.normal(0, sigma, size=n)
    for j, lam in enumerate(lams):
        A = X.T @ X + n * lam * np.eye(p)
        estimates[j, m] = np.linalg.solve(A, X.T @ y)

mean_est = estimates.mean(axis=1)                 # (n_lam, p)
bias2    = ((mean_est - theta) ** 2).sum(axis=1)
var_     = estimates.var(axis=1).sum(axis=1)
mse      = bias2 + var_

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(lams, bias2, "o-", color="steelblue", label="Bias²")
ax.plot(lams, var_,  "o-", color="orange",    label="Variance")
ax.plot(lams, mse,   "o-", color="crimson",   label="MSE = Bias² + Var")
ax.axhline(mse[0], color="black", ls=":", lw=1, label="MSE at smallest λ (≈ OLS)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("λ (log)")
ax.set_ylabel("error component (log)")
ax.set_title("Hoerl–Kennard in pictures: MSE drops below OLS for an open range of λ")
ax.legend(fontsize=8)
plt.show()

lam_best = lams[np.argmin(mse)]
print(f"Best λ (empirical) ≈ {lam_best:.3e}")
print(f"MSE at smallest λ  ≈ {mse[0]:.4f}")
print(f"MSE at best λ      ≈ {mse.min():.4f}    (ratio: {mse.min()/mse[0]:.2f})")

**Reading.**

- Bias^2 rises monotonically with $\lambda$.
- Variance falls monotonically with $\lambda$.
- Total MSE is U-shaped, with a strict interior minimum. The ratio at the bottom tells you how much ridge bought you compared to OLS.

## 4. Choosing $\lambda$ in practice

We need a data-driven rule. Three standard options.

### 4.1 k-fold cross-validation

Algorithmically identical to `02_polynomial_regression/04_statistics.ipynb` §3.1, replacing the *degree* sweep by a *$\lambda$* sweep:

```
ALGORITHM:  k-fold CV for ridge $\lambda$

Input:  X, y, candidate values  $\lambda_{1}$, ..., $\lambda_{M}$, folds k.
Output: $\lambda$\* minimising estimated test error.

1.  Partition data into k folds.
2.  for each candidate $\lambda$:
3.      for each fold j:
4.          fit ridge($\lambda$)  using all but fold j
5.          evaluate MSE on fold j
6.      CV($\lambda$)  $\leftarrow$  average fold MSE
7.  return  $\arg\min_{\lambda}$  CV($\lambda$).
```

Cost: M $\times$ k fits. Each fit is a Θ(n p^2 + p^3) Cholesky solve — usually fast for moderate p. With the path trick from `03_optimization.ipynb` §4 (re-use G across $\lambda$ within a fold), the cost drops to one factorisation per fold per $\lambda$ instead of one full re-fit.

### 4.2 Closed-form leave-one-out CV (PRESS statistic)

Ridge is a *linear* estimator: $\hat{y}_{ridge}$ = $H_{\lambda}$ y with $H_{\lambda}$ depending only on X and $\lambda$, not on y. For any linear smoother the LOO-CV error has a magic closed form — *no* extra fits required.

> **Theorem 4.2 (PRESS).** Let $H_{\lambda}$ be the ridge hat matrix and $\hat{r}_{i}$ = $y_i$ - [$H_{\lambda}$ y]_i the i-th fitted residual. Then
>
> LOO-MSE($\lambda$)  =  (1/n) $\cdot$ $\sum_{i=1}^{n}$  ( $\hat{r}_{i}$ / (1 - [$H_{\lambda}$]_{ii}) )^2.   (4.1)

**Proof sketch.** Define $\hat{\theta}_{\lambda}^{(-i)}$ as the ridge fit using all but the i-th observation. A classical update formula ("leave-one-out identity") for linear smoothers gives

```
$y_i$ - $x_i^T$ $\hat{\theta}_{\lambda}^{(-i)}$  =  $\hat{r}_{i}$ / ( 1 - [$H_{\lambda}$]_{ii} ).
```

Squaring and averaging over i gives (4.1). See `04_statistics.ipynb` §3.2 of the polynomial folder for the same trick under the OLS hat matrix; ridge is the regularised version. ∎

**Why this matters.** A naive LOO-CV needs n separate fits. With (4.1) we fit *once* per $\lambda$ on the full dataset, read the diagonal of $H_{\lambda}$, plug in. For an entire path of $\lambda$ values this is the cheapest cross-validation you will ever run.

### 4.3 Generalised cross-validation (GCV)

If even the diagonal of $H_{\lambda}$ is expensive to compute (large n), replace [$H_{\lambda}$]_{ii} by its average trace($H_{\lambda}$) / n = df($\lambda$) / n (eq. 4.1 of `02_mathematics.ipynb`). This gives **GCV**:

> GCV($\lambda$)  :=  (1/n) $\cdot$ $\sum_{i}$ $\hat{r}_{i}$^2 / ( 1 - df($\lambda$) / n )^2  =  RSS($\lambda$) / ( n - df($\lambda$) )^2.   (4.2)

GCV is the standard model-selection criterion in the smoothing-spline literature (Wahba). It is rotation-invariant (unlike LOO, which is not) and behaves robustly when the diagonal entries of $H_{\lambda}$ vary a lot.

In [ ]:
# Demo: closed-form LOO-CV (PRESS) on a single dataset; compare against
# a brute-force LOO loop on a coarse λ grid.
y = X @ theta + rng.normal(0, sigma, size=n)

def loo_press(lam):
    A = X.T @ X + n * lam * np.eye(p)
    H = X @ np.linalg.solve(A, X.T)
    theta_hat = np.linalg.solve(A, X.T @ y)
    r = y - X @ theta_hat
    diag_H = np.diag(H)
    return float(np.mean((r / (1 - diag_H)) ** 2))

def loo_brute(lam):
    errs = np.empty(n)
    for i in range(n):
        mask = np.ones(n, dtype=bool); mask[i] = False
        A = X[mask].T @ X[mask] + (n - 1) * lam * np.eye(p)
        theta_hat_minus_i = np.linalg.solve(A, X[mask].T @ y[mask])
        errs[i] = (y[i] - X[i] @ theta_hat_minus_i) ** 2
    return float(np.mean(errs))

lams_check = [1e-3, 1e-1, 1.0]
print(f"{'lambda':>10}   {'PRESS':>10}   {'brute LOO':>10}")
for lam in lams_check:
    print(f"{lam:>10.3f}   {loo_press(lam):>10.4f}   {loo_brute(lam):>10.4f}")

**Reading.** PRESS matches brute-force LOO to several decimals — at a fraction of the cost (n fits versus 1 fit). The tiny gap is from the brute-force version refitting on n - 1 points (so n $\cdot$ $\lambda$ in PRESS corresponds to (n - 1) $\cdot$ $\lambda_{LOO}$ inside the loop), not numerical error. In production we use (4.1) and never run the LOO loop directly.

## 5. The effective degrees of freedom in pictures

From `02_mathematics.ipynb` (4.1), df($\lambda$) = $\sum_{j}$ $\sigma_j$^2 / ($\sigma_j$^2 + n $\lambda$). This is the right "complexity counter" for ridge — it interpolates smoothly between $\text{rank}(X)$ at $\lambda$ = 0 and 0 at $\lambda$ → $\infty$.

In [ ]:
_, sigvals, _ = np.linalg.svd(X, full_matrices=False)
lams_plot = np.logspace(-5, 4, 80)
df_curve = np.array([np.sum(sigvals**2 / (sigvals**2 + n * lam)) for lam in lams_plot])

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.semilogx(lams_plot, df_curve, color="steelblue")
ax.axhline(p, color="black", ls=":", label=f"rank(X) = {p} (OLS limit)")
ax.axhline(0, color="black", ls=":", label="large-λ limit")
ax.set_xlabel("λ (log)")
ax.set_ylabel("effective dof  df(λ)")
ax.set_title("df(λ) = Σ σⱼ²/(σⱼ² + n λ)")
ax.legend(fontsize=8)
plt.show()
print(f"At λ = 0    :  df = {p}")
print(f"At λ = 0.1  :  df ≈ {df_curve[np.argmin(np.abs(lams_plot - 0.1))]:.2f}")
print(f"At λ = 10   :  df ≈ {df_curve[np.argmin(np.abs(lams_plot - 10))]:.2f}")

**Reading.** df is a smooth knob, not the discrete integer that OLS / polynomial regression had. "How many parameters does my ridge model use?" has a precise, $\lambda$-dependent answer.

## Takeaway

- **Sampling identity (1.2).**   $\hat{\theta}_{ridge}$ = $A_{\lambda}$ $\theta$ + ($X^T X$ + n $\lambda$ $I_p$)^{-1} Xᵀ $\varepsilon$. Bias and variance both flow from this.
- **Bias (Theorem 2.1).**   Bias($\hat{\theta}_{ridge}$) = - n $\lambda$ $\cdot$ ($X^T X$ + n $\lambda$ $I_p$)^{-1} $\theta$. Non-zero whenever $\lambda$ > 0 and $\theta$ $\neq$ 0; always points back toward the origin.
- **Variance (Theorems 2.2–2.3).**   Var($\hat{\theta}_{ridge}$) ⪯ Var($\hat{\theta}_{OLS}$) coordinate-wise in the SVD basis. The ratio equals the squared shrinkage factor $\rho_{j}$($\lambda$)^2.
- **Hoerl–Kennard (Theorem 3.2).**   For every X (full rank) and every $\theta$, $\sigma^2$ > 0, there exists $\lambda$\* > 0 with MSE($\lambda$\*) < MSE(0). Ridge always beats OLS in MSE for some $\lambda$.
- **$\lambda$ selection.**   k-fold CV (cheap), closed-form LOO via PRESS = (1/n) $\sum$ ($\hat{r}_{i}$ / (1 - H_{ii}))^2 (cheaper), GCV via RSS / (n - df($\lambda$))^2 (cheapest).
- **df($\lambda$)** is the right complexity counter; smoothly interpolates between $\text{rank}(X)$ and 0.

Next: `05_hands_on_programming.ipynb` — code up `RidgeRegression` from scratch with the Cholesky path solver, the closed-form LOO-CV, and cross-check against `sklearn.linear_model.Ridge` and `RidgeCV` on the diabetes dataset.